In [ ]:
pip install pinecone

In [ ]:
from pinecone import Pinecone, ServerlessSpec

In [ ]:
documents = [
    {
        "id": "doc-001",
        "text": "Pinecone is a fully managed vector database for search and recommendation.",
        "category": "documentation",
        "tag": "pinecone",
        "difficulty": "beginner",
        "url": "https://example.com/pinecone-intro"
    },
    {
        "id": "doc-002",
        "text": "To use Pinecone with Python, you create an index and upsert vectors with metadata.",
        "category": "documentation",
        "tag": "python",
        "difficulty": "beginner",
        "url": "https://example.com/pinecone-python"
    },
    {
        "id": "doc-003",
        "text": "Vector databases store embeddings that capture semantic meaning for semantic search.",
        "category": "blog",
        "tag": "vector-db",
        "difficulty": "intermediate",
        "url": "https://example.com/vector-db-concepts"
    },
    {
        "id": "doc-004",
        "text": "You can filter Pinecone search results using metadata such as category or difficulty.",
        "category": "faq",
        "tag": "metadata",
        "difficulty": "beginner",
        "url": "https://example.com/pinecone-metadata"
    },
    {
        "id": "doc-005",
        "text": "In Retrieval-Augmented Generation, a vector database like Pinecone stores document chunks.",
        "category": "blog",
        "tag": "rag",
        "difficulty": "intermediate",
        "url": "https://example.com/rag-pinecone"
    }
]


In [ ]:
pc = Pinecone(api_key="pcsk_5CCmmc_7fHrn24veNKT3khaZ35WGTf2WxCdAyi4LyVDF5yLtPHLxSoF8rg8YZ1pmS2b9pw",ssl_verify=False)

In [ ]:
pc

In [ ]:
import requests
import numpy as np
from typing import List, Union

EURON_API_KEY = "euri-eca149f81167bfd14dcbf6025b8bb834fec9aa6b728b4e8bac19531047ecb9f0"

def generate_embeddings(texts: Union[str, List[str]]):
    # Always make the input a list
    if isinstance(texts, str):
        texts = [texts]

    url = "https://api.euron.one/api/v1/euri/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {EURON_API_KEY}"
    }
    payload = {
        "input": texts,
        "model": "text-embedding-3-small"
    }

    response = requests.post(url, headers=headers, json=payload, verify=False)
    data = response.json()

    # Convert each embedding to numpy array
    embeddings = [np.array(item["embedding"], dtype=np.float32) for item in data["data"]]

    # Return single vector OR batch of vectors
    return embeddings[0] if len(embeddings) == 1 else np.stack(embeddings)


In [ ]:
INDEX_NAME = "sunil-pinecone-demo"

In [ ]:
pc.list_indexes()

In [ ]:
pc.create_index(
    name = INDEX_NAME,
    dimension = 1536,
    metric = "cosine",
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"))

In [ ]:
documents

In [ ]:
texts = [doc['text'] for doc in documents]
texts


In [ ]:
doc_embeddings = generate_embeddings(texts)
doc_embeddings

In [ ]:
zip(documents, doc_embeddings)

In [ ]:
l=[4,5,6]
l1 = ["Abha","Smitha","Kajal","Sonali"]
list(zip(l,l1))

In [ ]:
vector_to_upsert = []
for doc,emb in  zip(documents, doc_embeddings):
    metadata = {
        "category": doc["category"],
        "tag": doc["tag"],
        "difficulty": doc["difficulty"],
        "url": doc["url"],
        "text": doc["text"]
        }
    vector_item = {
        "id": doc["id"],
        "values": emb.tolist(),
        "metadata": metadata
    }
    
    vector_to_upsert.append(vector_item)
vector_to_upsert[0]    

In [ ]:
pc.Index(INDEX_NAME).upsert(vectors=vector_to_upsert)

In [ ]:
query = "How do I use Pinecone with Python?"
query_embedding = generate_embeddings(query)
pi_response = pc.Index(INDEX_NAME).query(vector=query_embedding.tolist(), top_k=3, include_metadata=True)
pi_response
